# Build ASL Fingerspelling LMDB (Kaggle, CPU)

Rebuilds `shravnchandr/asl-fs-lmdb` for CTC pre-training (`pretrain_fingerspelling.py`).
Same layout and preprocessing as the repo's `islr/core/data/build_fingerspelling_lmdb.py`
and the ASL Signs notebook (`build_asl_signs_lmdb.ipynb`):

- 393 columns = `config.ALL_COLUMNS` (LH, pose, RH, eyebrows + mouth; x, y, z)
- body-relative: nose → shoulder centre → hip centre → 0 fallback, per frame
- missing landmarks stored as 0 (the training code's `hand_presence()` flags and fills them)
- keys `fs:{sequence_id}` (float32 tensor bytes) and `fs:len:{sequence_id}`

Resume-safe: re-running skips sequences already written. Expected size ≈ 18 GB for the
67,208 train sequences (measured 263 MB per 998), so the LMDB is built outside
`/kaggle/working` when that is short of space and uploaded directly with the Kaggle CLI.

**Before running:** add the `asl-fingerspelling` competition data as input. Run on CPU;
*Save Version → Save & Run All* lets it run unattended (sessions are capped at 12 h).

In [ ]:
!pip install lmdb

In [ ]:
import io
import json
import multiprocessing as mp
import os
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import lmdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm import tqdm

## Settings

In [ ]:
DATA_DIR = Path("/kaggle/input/competitions/asl-fingerspelling")
DATASET_ID = "shravnchandr/asl-fs-lmdb"
DATASET_TITLE = "ASL Fingerspelling LMDB"

# Supplemental set (supplemental_metadata.csv): more sequences, ~1.8x the disk and time.
# Off = the dataset the training code was written against.
INCLUDE_SUPPLEMENTAL = False

NUM_WORKERS = os.cpu_count() or 4   # one parquet file per worker
MAP_SIZE_GB = 64                    # LMDB address space (sparse; not disk usage)
UPLOAD = True                       # publish to DATASET_ID when the build succeeds

# Build where there is room: the LMDB alone is ~18 GB (train only). /kaggle/working
# output is capped (20 GB), so prefer a scratch disk; the upload reads it from there.
NEED_GB = 22 if not INCLUDE_SUPPLEMENTAL else 40
candidates = [Path("/kaggle/tmp"), Path("/tmp"), Path("/kaggle/working")]
free = {}
for c in candidates:
    try:
        c.mkdir(parents=True, exist_ok=True)
        free[c] = shutil.disk_usage(c).free / 1024**3
    except OSError:
        pass
for c, gb in free.items():
    print(f"  {str(c):18s} {gb:6.1f} GB free")
WORK_ROOT = max(free, key=free.get)
assert free[WORK_ROOT] >= NEED_GB, (
    f"Only {free[WORK_ROOT]:.0f} GB free (need ~{NEED_GB} GB) — "
    "turn INCLUDE_SUPPLEMENTAL off or free disk"
)
DS_DIR = WORK_ROOT / "fs-lmdb-dataset"
DS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Building in {DS_DIR}")

## Column layout (must equal `islr/core/config.py`)

In [ ]:
INCLUDE_FACE = True
INCLUDE_DEPTH = True

FACE_LANDMARK_INDICES = {
    "left_eyebrow":  [70, 63, 105, 66, 107, 55, 65, 52],
    "right_eyebrow": [300, 293, 334, 296, 336, 285, 295, 282],
    "mouth_outer": [
        61, 146, 91, 181, 84, 17, 314, 405, 321, 375,
        291, 409, 270, 269, 267, 0, 37, 39, 40, 185,
    ],
    "mouth_inner": [
        78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
        308, 324, 318, 402, 317, 14, 87, 178, 88, 95,
    ],
}

SELECTED_FACE_INDICES: list[int] = []
for _face_idxs in FACE_LANDMARK_INDICES.values():
    SELECTED_FACE_INDICES.extend(_face_idxs)

AXES = ["x", "y", "z"] if INCLUDE_DEPTH else ["x", "y"]
ALL_COLUMNS: list[str] = []
for _lm_type, _count in [("left_hand", 21), ("pose", 33), ("right_hand", 21)]:
    for _i in range(_count):
        for _ax in AXES:
            ALL_COLUMNS.append(f"{_lm_type}_{_i}_{_ax}")
if INCLUDE_FACE:
    for _fi in SELECTED_FACE_INDICES:
        for _ax in AXES:
            ALL_COLUMNS.append(f"face_{_fi}_{_ax}")

COORD_FEAT: int = len(ALL_COLUMNS)
assert COORD_FEAT == 393, COORD_FEAT

# Fingerspelling parquets are wide: "{axis}_{type}_{idx}", e.g. "x_left_hand_0".
FS_COLS: list[str] = []
for _col in ALL_COLUMNS:
    _lm_type, _lm_idx, _axis = _col.rsplit("_", 2)
    FS_COLS.append(f"{_axis}_{_lm_type}_{_lm_idx}")


def lmdb_key(sequence_id: int) -> bytes:
    return f"fs:{sequence_id}".encode()


def lmdb_length_key(sequence_id: int) -> bytes:
    return f"fs:len:{sequence_id}".encode()

## Worker: one parquet file → all its pending sequences

In [ ]:
def _process_parquet(args: tuple) -> list[tuple]:
    """Returns [(sequence_id, tensor_bytes | None, length | None, error | None)].

    Reads only the 393 needed columns (+ frame) instead of all ~1,630 — the
    origin landmarks (pose 0, 11, 12, 23, 24) are among them. Normalisation is
    identical to the repo builder: per frame and axis, subtract the nose, else
    the shoulder centre, else the hip centre, else 0; NaN (missing) → 0.
    """
    import io
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq
    import torch

    parquet_path, pending, fs_cols, axes = args
    try:
        schema = set(pq.read_schema(parquet_path).names)
        cols = ["frame"] + [c for c in fs_cols if c in schema]
        wide = pd.read_parquet(parquet_path, columns=cols)
    except Exception as e:
        return [(sid, None, None, f"read {parquet_path}: {e}") for sid in pending]

    data_cols = [c for c in cols if c != "frame"]
    col_index = {c: i for i, c in enumerate(data_cols)}
    out_index = [col_index.get(c) for c in fs_cols]

    results = []
    for sid in pending:
        try:
            seq = wide.loc[[sid]].sort_values("frame")
            raw = seq[data_cols].to_numpy(dtype=np.float32)
            T = len(raw)
            for ax in axes:
                def get(name):
                    i = col_index.get(f"{ax}_{name}")
                    return raw[:, i] if i is not None else np.full(T, np.nan, np.float32)

                nose = get("pose_0")
                shoulder = (get("pose_11") + get("pose_12")) / 2
                hip = (get("pose_23") + get("pose_24")) / 2
                origin = np.where(
                    ~np.isnan(nose), nose,
                    np.where(~np.isnan(shoulder), shoulder,
                             np.where(~np.isnan(hip), hip, 0.0)),
                )
                ax_idx = [i for c, i in col_index.items() if c.startswith(f"{ax}_")]
                raw[:, ax_idx] -= origin[:, None]

            out = np.zeros((T, len(fs_cols)), dtype=np.float32)
            for fi, ci in enumerate(out_index):
                if ci is not None:
                    v = raw[:, ci]
                    out[:, fi] = np.where(np.isnan(v), 0.0, v)

            buf = io.BytesIO()
            torch.save(torch.from_numpy(out), buf)
            results.append((sid, buf.getvalue(), T, None))
        except Exception as e:
            results.append((sid, None, None, str(e)))
    return results

## Build (resume-safe)

In [ ]:
_WRITE_BATCH = 500


def load_metadata() -> pd.DataFrame:
    df = pd.read_csv(DATA_DIR / "train.csv")
    if INCLUDE_SUPPLEMENTAL:
        sup = pd.read_csv(DATA_DIR / "supplemental_metadata.csv")
        df = pd.concat([df, sup], ignore_index=True)
    assert df["sequence_id"].is_unique, "duplicate sequence_id across metadata files"
    return df


def build(lmdb_path: Path, allow_errors: bool = False) -> pd.DataFrame:
    meta = load_metadata()
    total = len(meta)
    print(
        f"FS LMDB: {total:,} sequences, {meta['path'].nunique()} parquet files, "
        f"{meta['participant_id'].nunique()} signers → {lmdb_path}  (workers={NUM_WORKERS})"
    )
    lmdb_path.mkdir(parents=True, exist_ok=True)
    env = lmdb.open(str(lmdb_path), map_size=MAP_SIZE_GB * 1024**3)

    written = skipped = errors = 0
    pending: set[int] = set()
    with env.begin() as txn:
        for sid in meta["sequence_id"]:
            if txn.get(lmdb_key(int(sid))) is not None:
                skipped += 1
            else:
                pending.add(int(sid))

    tasks = []
    for rel, g in meta.groupby("path", sort=False):
        sids = [int(s) for s in g["sequence_id"] if int(s) in pending]
        if sids:
            tasks.append((str(DATA_DIR / rel), sids, FS_COLS, AXES))
    print(f"Processing {len(pending):,} sequences in {len(tasks)} files ({skipped:,} already done)")

    buf: list = []

    def flush():
        with env.begin(write=True) as txn:
            for sid, data, length in buf:
                txn.put(lmdb_key(sid), data)
                txn.put(lmdb_length_key(sid), str(length).encode())
        buf.clear()

    t0 = time.time()
    with tqdm(total=total, desc="FS LMDB", unit="seq", mininterval=30) as pbar:
        pbar.update(skipped)
        # fork: workers inherit the notebook-defined _process_parquet (spawn /
        # forkserver — Python 3.14's Linux default — cannot import it from __main__).
        with ProcessPoolExecutor(max_workers=NUM_WORKERS, mp_context=mp.get_context("fork")) as ex:
            for results in ex.map(_process_parquet, tasks, chunksize=1):
                for sid, data, length, err in results:
                    if err:
                        errors += 1
                        tqdm.write(f"  ERROR seq {sid}: {err}", file=sys.stderr)
                    else:
                        buf.append((sid, data, length))
                        written += 1
                        if len(buf) >= _WRITE_BATCH:
                            flush()
                    pbar.update(1)
                pbar.set_postfix(written=written, errors=errors)
        if buf:
            flush()
    env.close()
    print(f"\nDone in {(time.time() - t0) / 60:.0f} min: written={written} skipped={skipped} errors={errors}")
    if errors and not allow_errors:
        raise RuntimeError(f"{errors} sequence(s) failed — re-run this cell to retry them")
    return meta


meta = build(DS_DIR / "fs.lmdb")

# `kaggle datasets` skips subdirectories: flatten to a single file (same filesystem → instant).
os.rename(DS_DIR / "fs.lmdb" / "data.mdb", DS_DIR / "fs.lmdb.mdb")
shutil.rmtree(DS_DIR / "fs.lmdb")
meta[["sequence_id", "participant_id", "phrase"]].to_csv(DS_DIR / "train.csv", index=False)
shutil.copy(DATA_DIR / "character_to_prediction_index.json", DS_DIR)
print(f"LMDB size: {(DS_DIR / 'fs.lmdb.mdb').stat().st_size / 1024**3:.1f} GB")

## Verify (same checks as `scripts/setup_runpod.sh --with-fs`, but every key)

In [ ]:
env = lmdb.open(str(DS_DIR / "fs.lmdb.mdb"), subdir=False, readonly=True, lock=False)
with env.begin() as txn:
    missing = [s for s in meta["sequence_id"] if txn.get(lmdb_key(int(s))) is None]
    sample = [torch.load(io.BytesIO(bytes(txn.get(lmdb_key(int(s))))), weights_only=True)
              for s in meta["sequence_id"][:1000]]
    lengths = [int(txn.get(lmdb_length_key(int(s)))) for s in meta["sequence_id"][:1000]]
env.close()
assert not missing, f"{len(missing)} keys missing, e.g. {missing[:5]}"
assert all(x.ndim == 2 and x.shape[1] == COORD_FEAT for x in sample), "wrong sample width"
assert [len(x) for x in sample] == lengths, "length keys disagree with tensors"
RH = 54 * 3  # right hand starts after LH (21) + pose (33) landmarks
lh_any = np.mean([bool((x[:, :63] != 0).any()) for x in sample])
rh_any = np.mean([bool((x[:, RH:RH + 63] != 0).any()) for x in sample])
print(f"All {len(meta):,} keys present | first 1000: frames median {int(np.median(lengths))}, "
      f"max {max(lengths)} | clips with a detected left hand {lh_any:.0%}, right hand {rh_any:.0%}")

## Upload as a Kaggle dataset version (or create it)

In [ ]:
if UPLOAD:
    (DS_DIR / "dataset-metadata.json").write_text(json.dumps({
        "title": DATASET_TITLE,
        "id": DATASET_ID,
        "licenses": [{"name": "other"}],
    }))
    msg = f"Rebuild: {len(meta):,} sequences, COORD_FEAT {COORD_FEAT}" + (
        " (+ supplemental)" if INCLUDE_SUPPLEMENTAL else "")
    # Upload only the dataset files, not a stale fs.lmdb directory.
    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(DS_DIR), "-m", msg],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:  # dataset does not exist yet → create it
        r = subprocess.run(["kaggle", "datasets", "create", "-p", str(DS_DIR)],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)
    r.check_returncode()  # fail loudly: a finished build must not look uploaded when it wasn't